In [36]:
#Setup and oLLAMA authentication

import json
import requests
from bs4 import BeautifulSoup
import re
import time

# Load credentials
with open('Ollama_Creds.json', 'r') as f:
    creds = json.load(f)
username = creds['User']
password = creds['Password']

ollama_base = "https://ollama.lourie.info"
auth = (username, password)

def query_ollama(prompt, model="qwen3.5:9b", max_tokens=1000, timeout=90):

    """
    Send a prompt to Ollama and return the generated text.
    Args:
        prompt (str): The input prompt.
        model (str): Model name (check available models with list_models()).
        max_tokens (int): Maximum number of tokens to generate.
        timeout (int): Seconds to wait for the response (default 15).
    Returns:
        str or None: The generated text, or None if request fails/times out.
    """
    """
    Send a prompt to Ollama and return the generated text.
    Enhanced with timing and detailed error logging.
    """
    """
    Send a prompt to Ollama and return the generated text.
    Enhanced with timing and detailed error logging.
    """
    url = f"{ollama_base}/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "num_predict": max_tokens
        }
    }
    start_time = time.time()
    try:
        response = requests.post(url, json=payload, auth=auth, timeout=timeout)
        elapsed = time.time() - start_time
        print(f"Ollama response received in {elapsed:.2f} seconds with status {response.status_code}")
        
        if response.status_code == 200:
            # Print the raw response text for debugging
            print("Raw response text (first 500 chars):", response.text[:500])
            result = response.json()
            print("Parsed JSON:", result)
            generated = result.get('response', '').strip()
            print(f"Extracted response: '{generated}'")
            return generated
        else:
            print(f"Ollama error: {response.status_code} - {response.text[:200]}")
            return None
            
    except requests.exceptions.Timeout:
        elapsed = time.time() - start_time
        print(f"Ollama request timed out after {elapsed:.2f} seconds (timeout={timeout}s).")
        return None
    except requests.exceptions.ConnectionError as e:
        elapsed = time.time() - start_time
        print(f"Ollama connection error after {elapsed:.2f} seconds: {e}")
        return None
    except Exception as e:
        elapsed = time.time() - start_time
        print(f"Unexpected error after {elapsed:.2f} seconds: {e}")
        return None
    

def list_models():
    """List available models on the Ollama server."""
    url = f"{ollama_base}/api/tags"
    response = requests.get(url, auth=auth)
    if response.status_code == 200:
        return [m['name'] for m in response.json().get('models', [])]
    else:
        print(f"Failed to list models: {response.status_code}")
        return []

In [37]:
#Extract description

def extract_product_name(user_input):
    """Use LLM to extract the product name from a Russian user query."""
    print("Шаг 1: Используем LLM для сокращения поискового запроса.")
    prompt = f"Извлеки название товара из запроса пользователя на русском языке. Ответь только названием товара, без лишних слов, но добавь глагол 'купить', чтобы в поисковой выдаче были релевантные ссылки. Запрос: '{user_input}'"
    product = query_ollama(prompt, max_tokens=200, timeout=90)
    if not product:
        # Fallback: return the original input if LLM fails
        print("LLM answer taking too long, invoking bypas...")
        product = user_input.strip() + "купить"
    print(f"Используя LLM, преобразуем поисковый запрос в {product}.")
    return product

# Test
print(extract_product_name("Я кочу купить скатрть"))  # Should print "скатерть" (correcting typo)

Шаг 1: Используем LLM для сокращения поискового запроса.
Ollama response received in 57.69 seconds with status 200
Raw response text (first 500 chars): {"model":"qwen3.5:9b","created_at":"2026-03-16T13:41:02.365217612Z","response":"","thinking":"Thinking Process:\n\n1.  **Analyze the Request:**\n    *   Task: Extract the product name from the user's query in Russian.\n    *   Constraint 1: Respond *only* with the product name.\n    *   Constraint 2: Add the verb 'купить' (buy) to make search results relevant.\n    *   Input Query: 'Я кочу купить скатрть' (This looks like a typo-heavy sentence).\n    *   Goal: Identify the intended product and f
Parsed JSON: {'model': 'qwen3.5:9b', 'created_at': '2026-03-16T13:41:02.365217612Z', 'response': '', 'thinking': 'Thinking Process:\n\n1.  **Analyze the Request:**\n    *   Task: Extract the product name from the user\'s query in Russian.\n    *   Constraint 1: Respond *only* with the product name.\n    *   Constraint 2: Add the verb \'купить\' 

In [ ]:
def search_yandex(query, num_results=20):
    """Search ya.ru and return top result items with title, URL, and snippet."""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    url = f"https://ya.ru/search?text={requests.utils.quote(query)}&lr=213"
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        print(f"Search failed with status {response.status_code}")
        return []
    
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # DEBUG: print a small sample of the HTML to see the structure
    """
    print("--- First 1000 chars of HTML ---")
    print(soup.prettify()[:1000])
    print("--------------------------------")
    """
    
    # Try multiple common Yandex result selectors
    possible_selectors = [
        'li.serp-item',
        'div.serp-item',
        'div.organic',
        'article',
        'div[class*="organic"]',
        'div[class*="serp-item"]',
        '.main__content li',
        '.search-result',
        '.result'
    ]
    
    items = []
    for selector in possible_selectors:
        items = soup.select(selector)
        if items:
            print(f"Found {len(items)} items with selector: '{selector}'")
            break
    
    if not items:
        # Last resort: heuristic – any div that contains a link and has substantial text
        all_divs = soup.find_all('div', recursive=True)
        potential = []
        for div in all_divs:
            if div.find('a', href=True) and len(div.get_text(strip=True)) > 50:
                potential.append(div)
        if potential:
            items = potential
            print(f"Heuristic found {len(items)} potential items")
        else:
            print("No search results found with any method.")
            return []
    
    results = []
    for item in items[:num_results]:
        # Title: first h2 or link text
        title_elem = (item.find('h2') or 
                      item.find('a', class_='link') or 
                      item.find('a', class_='organic__title') or 
                      item.find('a', href=True))
        title = title_elem.get_text(strip=True) if title_elem else ''
        
        # URL
        link = item.find('a', href=True)
        url = link['href'] if link else ''
        if url.startswith('/'):
            url = 'https://ya.ru' + url
        
        # Snippet: try known containers, else fallback to text without title
        snippet_elem = (item.find('div', class_='text-container') or 
                        item.find('div', class_='organic__text') or 
                        item.find('div', class_='serp-item__text') or
                        item.find('div', class_='description'))
        if snippet_elem:
            snippet = snippet_elem.get_text(strip=True)
        else:
            all_text = item.get_text(strip=True)
            if title:
                snippet = all_text.replace(title, '', 1).strip()
            else:
                snippet = all_text[:300]
        
        results.append({
            'title': title,
            'url': url,
            'snippet': snippet
        })
    return results

In [8]:
test_results = search_yandex("Теплица", num_results=5)
print(f"Found {len(test_results)} results")
if test_results:
    print(test_results[0])

--- First 1000 chars of HTML ---
<!DOCTYPE html>
<html lang="en">
 <head>
  <meta charset="utf-8"/>
  <meta content="width=device-width,initial-scale=1" name="viewport"/>
  <title>
   ÐÐµÑÐ¸ÑÐ¸ÐºÐ°ÑÐ¸Ñ
  </title>
  <link as="script" crossorigin="anonymous" href="/tmgrdfrend.fp.js" rel="preload"/>
  <script>
   !function(e){if(e.Ya=e.Ya||{},Ya.Rum)throw new Error("Rum: interface is already defined");var n=e.performance,t=n&&n.timing&&n.timing.navigationStart||Ya.startPageLoad||+new Date,i=e.requestAnimationFrame,r=function(){},o=Ya.Rum={_defTimes:[],_defRes:[],_countersToExposeAsEvents:["2325","2616.85.1928","react.inited"],enabled:!!n,version:"6.1.21",vsStart:document.visibilityState,vsChanged:!1,vsChangeTime:1/0,_deltaMarks:{},_markListeners:{},_onComplete:[],_onInit:[],_unsubscribers:[],_eventLisneters:{},_settings:{},_vars:{},init:function(e,n){o._settings=e,o._vars=n},getTime:n&&n.now?function(){return n.now()}:Date.now?function(){return Date.now()-t}:function(){return new Dat

In [12]:
%pip install ddgs

/usr/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=33505) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


Note: you may need to restart the kernel to use updated packages.


In [13]:
def search_yandex(query, num_results=20):
    """Search ya.ru and return top result items with title, URL, and snippet."""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'ru-RU,ru;q=0.9,en-US;q=0.8,en;q=0.7',
        'DNT': '1',
        'Connection': 'keep-alive',
        'Upgrade-Insecure-Requests': '1',
        'Sec-Fetch-Dest': 'document',
        'Sec-Fetch-Mode': 'navigate',
        'Sec-Fetch-Site': 'none',
        'Sec-Fetch-User': '?1',
        'Cache-Control': 'max-age=0',
    }
    session = requests.Session()
    # Set a basic Yandex cookie to appear more legitimate
    session.cookies.set('yandexuid', '1234567890123456789', domain='.ya.ru')
    url = f"https://ya.ru/search?text={requests.utils.quote(query)}&lr=213"
    
    try:
        response = session.get(url, headers=headers, timeout=10)
    except Exception as e:
        print(f"Yandex request error: {e}")
        return None
    
    if response.status_code != 200:
        print(f"Yandex search failed with status {response.status_code}")
        return None
    
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Check if it's a captcha page
    title_tag = soup.find('title')
    if title_tag and 'верификация' in title_tag.text.lower():
        print("Yandex returned a captcha page.")
        return None
    
    # Try to find result items using common selectors
    possible_selectors = [
        'li.serp-item',
        'div.serp-item',
        'div.organic',
        'article',
        'div[class*="organic"]',
        'div[class*="serp-item"]',
        '.main__content li',
        '.search-result',
        '.result'
    ]
    
    items = []
    for selector in possible_selectors:
        items = soup.select(selector)
        if items:
            print(f"Yandex: found {len(items)} items with selector '{selector}'")
            break
    
    if not items:
        # Heuristic: any div with a link and substantial text
        all_divs = soup.find_all('div', recursive=True)
        potential = []
        for div in all_divs:
            if div.find('a', href=True) and len(div.get_text(strip=True)) > 50:
                potential.append(div)
        if potential:
            items = potential
            print(f"Yandex heuristic found {len(potential)} potential items")
        else:
            print("Yandex: no results found.")
            return None
    
    results = []
    for item in items[:num_results]:
        title_elem = (item.find('h2') or 
                      item.find('a', class_='link') or 
                      item.find('a', class_='organic__title') or 
                      item.find('a', href=True))
        title = title_elem.get_text(strip=True) if title_elem else ''
        
        link = item.find('a', href=True)
        url = link['href'] if link else ''
        if url.startswith('/'):
            url = 'https://ya.ru' + url
        
        snippet_elem = (item.find('div', class_='text-container') or 
                        item.find('div', class_='organic__text') or 
                        item.find('div', class_='serp-item__text') or
                        item.find('div', class_='description'))
        if snippet_elem:
            snippet = snippet_elem.get_text(strip=True)
        else:
            all_text = item.get_text(strip=True)
            if title:
                snippet = all_text.replace(title, '', 1).strip()
            else:
                snippet = all_text[:300]
        
        results.append({
            'title': title,
            'url': url,
            'snippet': snippet
        })
    return results


def search_duckduckgo(query, num_results=20):
    """Search DuckDuckGo and return top result items."""
    try:
        #from duckduckgo_search import DDGS
        from ddgs import DDGS
    except ImportError:
        print("DuckDuckGo search library not installed. Please run: !pip install duckduckgo-search")
        return []
    
    with DDGS() as ddgs:
        results = []
        for r in ddgs.text(query, max_results=num_results):
            results.append({
                'title': r.get('title', ''),
                'url': r.get('href', ''),
                'snippet': r.get('body', '')
            })
        return results


def search(query, num_results=20, prefer='yandex'):
    """
    Main search function: tries Yandex first, if it fails falls back to DuckDuckGo.
    Returns a list of result dicts with 'title', 'url', 'snippet'.
    """
    if prefer == 'yandex':
        results = search_yandex(query, num_results)
        if results is not None and len(results) > 0:
            return results
        print("Falling back to DuckDuckGo...")
        return search_duckduckgo(query, num_results)
    else:
        return search_duckduckgo(query, num_results)

In [15]:
# Test the new search function
test_results = search("купить скатерть", num_results=5)
print(f"Found {len(test_results)} results")
if test_results:
    print(test_results[0])

Yandex: no results found.
Falling back to DuckDuckGo...
Found 5 results
{'title': 'Скатертьводоотталкивающаякупитьна OZON по низкой цене', 'url': 'https://www.OZON.ru/category/skatert-vodoottalkivayuschaya/', 'snippet': 'Скатертьводоотталкивающая –покупайтена OZON по выгодным ценам! Быстрая и бесплатная доставка, большой ассортимент, бонусы, рассрочка и кэшбэк.'}


In [17]:
def extract_price(text):
    """Extract a price in Russian rubles from text using regex."""
    # Patterns for common Russian price formats
    patterns = [
        r'(\d+(?:[.,\s]?\d+)?)\s*(?:₽|руб|рублей|р\.|руб\.)',  # 1 200 ₽, 1200 руб
        r'(\d+(?:[.,\s]?\d+)?)\s*(?:₽|руб)',                   # simpler variant
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            price_str = match.group(1).replace(' ', '').replace(',', '.').replace('\xa0', '')
            try:
                return float(price_str)
            except ValueError:
                continue
    return None

# Test
print(extract_price("Цена: 1 299 ₽"))          # 1299.0
print(extract_price("всего за 500 рублей"))    # 500.0
print(extract_price("скидка 20%"))              # None

1299.0
500.0
None


In [18]:
def process_results(results):
    """Convert raw search results into items with description, price, and URL."""
    items = []
    for res in results:
        full_text = f"{res['title']} {res['snippet']}"
        price = extract_price(full_text)
        # Truncate description to 300 characters
        description = (res['title'] + ' - ' + res['snippet'])[:300]
        items.append({
            'description': description,
            'price': price,
            'url': res['url']
        })
    return items

# Test with previous results
if test_results:
    test_items = process_results(test_results)
    for it in test_items[:3]:
        print(it)

{'description': 'Скатертьводоотталкивающаякупитьна OZON по низкой цене - Скатертьводоотталкивающая –покупайтена OZON по выгодным ценам! Быстрая и бесплатная доставка, большой ассортимент, бонусы, рассрочка и кэшбэк.', 'price': None, 'url': 'https://www.OZON.ru/category/skatert-vodoottalkivayuschaya/'}
{'description': 'Скатертикупитьпо выгодной цене в интернет-магазине HOFF.ru - Скатертикупитьв интернет-магазине HOFF. Быстрая доставка поМосквеи всей России. Широкий ассортимент — 87 товаров Самовывоз товаров для дома из наших...', 'price': None, 'url': 'https://hoff.ru/catalog/tovary_dlya_doma/tekstil/tekstil_dlya_kuhni/skaterti/'}
{'description': 'Силиконовыескатертии клеенкикупитьпо выгодной цене... - Силиконовыескатертии клеенкикупитьс бесплатной доставкой за 1 день. в интернет-магазине Магнит Маркет. Выгодные цены, рассрочка, гарантия!', 'price': None, 'url': 'https://mm.ru/category/silikonovye-skaterti-i-kleenki-15109'}


In [20]:
def find_cheapest(items):
    """Return the item with the lowest price (ignoring items without price)."""
    valid_items = [item for item in items if item['price'] is not None]
    if not valid_items:
        return None
    return min(valid_items, key=lambda x: x['price'])

# Test
cheapest = find_cheapest(test_items)
if cheapest:
    print(f"Cheapest: {cheapest['price']} - {cheapest['description']}")
else:
    print("No prices found.")

No prices found.


In [21]:
def agent(user_input):
    """Main agent: extract product, search, process, return cheapest item."""
    print(f"User input: {user_input}")
    
    # Step 2: extract product name
    product = extract_product_name(user_input)
    print(f"Extracted product: {product}")
    
    # Step 3: search Yandex
    results = search_yandex(product, num_results=20)
    print(f"Found {len(results)} search results")
    
    # Step 4-5: process and extract prices
    items = process_results(results)
    priced_count = len([i for i in items if i['price'] is not None])
    print(f"Processed {len(items)} items, {priced_count} with prices")
    
    # Step 6: find cheapest
    cheapest = find_cheapest(items)
    if cheapest:
        print(f"Cheapest item: {cheapest['price']} RUB")
        return cheapest
    else:
        print("No items with price found.")
        return {"description": "No prices found", "price": None, "url": None}

# Test the full agent
result = agent("Я кочу купить скатерть")
print("\nFinal result:")
print(result)

User input: Я кочу купить скатерть
Extracted product: скатерть
Yandex: no results found.


TypeError: object of type 'NoneType' has no len()